# Embedding RAG and Multi-Model Agents Using Amazon

In [10]:
# Install the missing package
!pip install tiktoken
!pip install openai

# Then import your packages
from importlib.metadata import version  # Used to check package versions
import tiktoken  # Library for tokenizing text (used by OpenAI models)
import torch     # PyTorch library for machine learning

# Print version information for key packages
print("torch version", version("torch"))
print("tiktoken version", version("tiktoken"))

  Using cached openai-2.29.0-py3-none-any.whl.metadata (29 kB)
Using cached openai-2.29.0-py3-none-any.whl (1.1 MB)
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.0 MB 3.2 MB/s eta 0:00:01
   -------------------- ------------------- 1.0/2.0 MB 3.0 MB/s eta 0:00:01
   ------------------------------- -------- 1.6/2.0 MB 2.7 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 2.5 MB/s  0:00:00

   --- ------------------------------------  1/13 [tqdm]
   --- ------------------------------------  1/13 [tqdm]
   --- ------------------------------------  1/13 [tqdm]
   --------- ------------------------------  3/13 [pydantic-core]
   --------------- ------------------------  5/13 [h11]
   --------------------- ------------------  7/13 [anyio]
   --------------------- ------------------  7/13 [anyio]
   --------------------- ------------------  7/13 [anyio]
   --------------------- ------------------  7

In [11]:
# Reading the whole file 
with open("Rathod.txt", "r",encoding="utf-8")as f:
    raw_text=f.read()
print("Total number of character:",len(raw_text))
print(raw_text[:144])

Total number of character: 144
Hii, my name is Ajay Rathod ,i will study in BE computer science and engineering, I also pursuving my degree in the upcoming month completely. 



* The goal is to tokenize this text for an LLM
* Lets develop a simple tokenizer based on some simple sample text  that we can then later apply to the above 

The following regular expression will split whitespaces

In [12]:
import re
text="Hello, world. This ia s test,"
result =re.split(r'(\s)',text)
print(result)

['Hello,', ' ', 'world.', ' ', 'This', ' ', 'ia', ' ', 's', ' ', 'test,']


In [13]:
text ="Hello, world. Is this-- a test?"
result=re.split(r'([,.:;?_"()\']|--|\s)',text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


Removing Whitespace or Not ?

* When developing a simple tokenizer , whether we should encode whitespace as characters or just remove on our application and its requirements.
* Removing whitespace reduces the memory and computing requirements.
* Here, we remove whitespace for simplicity and brevity of the tokenizer outputs. Later , we will switch to a tokenization scheme that include whitespaces.  

In [14]:
# Lets now tokenized our 'raw text' using the regex
preprocessed =re.split(r'([,.?!"()]\'|--|\s)',raw_text)
preprocessed =[item.strip() for item in preprocessed if item.strip()]

# Printing the first 30 tokens 
print(preprocessed[:30])


['Hii,', 'my', 'name', 'is', 'Ajay', 'Rathod', ',i', 'will', 'study', 'in', 'BE', 'computer', 'science', 'and', 'engineering,', 'I', 'also', 'pursuving', 'my', 'degree', 'in', 'the', 'upcoming', 'month', 'completely.']


Lets Calculated the total  number of tokens

In [15]:
print(len(preprocessed))

25


## Converting Tokens into IDs:
* Next, we convert the text tokens into IDs  that we can process via embedding layers later.

1. Tokenization breaks down the input text into individual tokens.
2. Each uniques token is added to the vocabulary in alphabetical order.

* From these tokens , we can now build a vocabulary that consists of all uniques tokens.
* This conversion is an intermediate step before converting the IDs into embedding vectors.

In [16]:
# converting Tokens into IDs:
# Finding the unique words/tokens 
all_words = sorted(set(preprocessed))
vocab_size =len(all_words)

# Printing the total vocabulary size
print(vocab_size)

23


In [17]:
vocab ={token:integer for integer,token  in enumerate(all_words)}